In [1]:
import polars as pl
import pandas as pd
from pathlib import Path
from glob import glob

DATA = Path("../data")
OUTPUTS = Path("../outputs")

submission = pd.read_csv("submission.csv")

print("Submission shape:", submission.shape)
submission.head()

Submission shape: (64062, 4)


,account_id,is_mule,suspicious_start,suspicious_end
0,ACCT_000005,0.002405,NaN,NaN
1,ACCT_000007,0.002405,NaN,NaN
2,ACCT_000009,0.002424,NaN,NaN
3,ACCT_000015,0.002405,NaN,NaN
4,ACCT_000016,0.002405,NaN,NaN


In [44]:
HIGH_RISK_THRESHOLD = 0.25

high_risk_accounts = submission[
    submission["is_mule"] > HIGH_RISK_THRESHOLD
]["account_id"]

print("High risk accounts:", len(high_risk_accounts))

High risk accounts: 1678


In [45]:
# Convert to Python set for fast filtering
high_risk_set = set(high_risk_accounts)

txn_files = sorted(glob("../data/transactions/batch-*/part_*.parquet"))

print("Total transaction files:", len(txn_files))

Total transaction files: 396


In [46]:
txn_list = []

for file in txn_files:
    
    df = (
        pl.scan_parquet(file)
        .filter(pl.col("account_id").is_in(high_risk_set))
        .select([
            "account_id",
            "transaction_timestamp",
            "amount",
            "txn_type"
        ])
        .collect()
    )
    
    if df.height > 0:
        txn_list.append(df)

print("Files containing high-risk activity:", len(txn_list))

Files containing high-risk activity: 387


In [47]:
txns = pl.concat(txn_list)

print("Filtered transaction shape:", txns.shape)

txns = txns.with_columns(
    pl.col("transaction_timestamp").str.to_datetime(
        format="%Y-%m-%dT%H:%M:%S",
        strict=False
    ).alias("ts")
)

txns.head()

Filtered transaction shape: (3445867, 4)


account_id,transaction_timestamp,amount,txn_type,ts
str,str,f64,str,datetime[μs]
"""ACCT_000116""","""2020-07-02T12:20:57""",1000.0,"""C""",2020-07-02 12:20:57
"""ACCT_000116""","""2020-07-03T13:46:20""",469.25,"""C""",2020-07-03 13:46:20
"""ACCT_000116""","""2020-07-06T15:13:18""",181.72,"""C""",2020-07-06 15:13:18
"""ACCT_000116""","""2020-07-07T17:46:51""",3191.42,"""D""",2020-07-07 17:46:51
"""ACCT_000116""","""2020-07-09T09:21:23""",43.8,"""D""",2020-07-09 09:21:23


In [48]:
txns = txns.sort(["account_id", "ts"])

print("Sorted transactions ready.")

Sorted transactions ready.


In [49]:
txns = txns.with_columns(
    (
        pl.col("ts")
        .diff()
        .over("account_id")
        .dt.total_seconds()
    ).alias("gap_seconds")
)

txns.head()

account_id,transaction_timestamp,amount,txn_type,ts,gap_seconds
str,str,f64,str,datetime[μs],i64
"""ACCT_000116""","""2020-07-02T12:20:57""",1000.0,"""C""",2020-07-02 12:20:57,null
"""ACCT_000116""","""2020-07-03T13:46:20""",469.25,"""C""",2020-07-03 13:46:20,91523
"""ACCT_000116""","""2020-07-06T15:13:18""",181.72,"""C""",2020-07-06 15:13:18,264418
"""ACCT_000116""","""2020-07-07T17:46:51""",3191.42,"""D""",2020-07-07 17:46:51,95613
"""ACCT_000116""","""2020-07-09T09:21:23""",43.8,"""D""",2020-07-09 09:21:23,142472


In [50]:
SESSION_GAP = 86400  # 24 hours

txns = txns.with_columns(
    (
        (pl.col("gap_seconds") > SESSION_GAP)
        .fill_null(True)
        .cast(pl.Int32)
        .cum_sum()
        .over("account_id")
    ).alias("session_id")
)

In [51]:
txns.select(["account_id","ts","gap_seconds","session_id"]).head(20)

account_id,ts,gap_seconds,session_id
str,datetime[μs],i64,i32
"""ACCT_000116""",2020-07-02 12:20:57,null,1
"""ACCT_000116""",2020-07-03 13:46:20,91523,2
"""ACCT_000116""",2020-07-06 15:13:18,264418,3
"""ACCT_000116""",2020-07-07 17:46:51,95613,4
"""ACCT_000116""",2020-07-09 09:21:23,142472,5
…,…,…,…
"""ACCT_000116""",2020-07-17 21:44:59,107660,9
"""ACCT_000116""",2020-07-19 07:03:08,119889,10
"""ACCT_000116""",2020-07-20 12:32:22,106154,11


In [52]:
sessions = (
    txns
    .group_by(["account_id", "session_id"])
    .agg([
        pl.col("ts").min().alias("start"),
        pl.col("ts").max().alias("end"),
        pl.len().alias("txn_count"),
        pl.col("amount").sum().alias("total_amount"),
        (pl.col("txn_type") == "C").sum().alias("credits"),
        (pl.col("txn_type") == "D").sum().alias("debits")
    ])
)

sessions.head()

account_id,session_id,start,end,txn_count,total_amount,credits,debits
str,i32,datetime[μs],datetime[μs],u32,f64,u32,u32
"""ACCT_170125""",211,2022-04-13 17:00:36,2022-04-14 19:31:26,3,2009.6,1,2
"""ACCT_002877""",591,2025-01-30 09:49:32,2025-01-30 09:49:32,1,4298.48,1,0
"""ACCT_165595""",192,2022-03-15 10:49:20,2022-03-18 01:41:54,7,205434.48,3,4
"""ACCT_175795""",151,2023-06-30 18:26:34,2023-06-30 18:26:34,1,75100.0,0,1
"""ACCT_179899""",284,2023-08-05 23:51:38,2023-08-12 10:28:31,20,248167.74,13,7


In [53]:
sessions = sessions.with_columns(
(
    pl.col("txn_count") * 0.45 +
    pl.col("total_amount").abs() * 0.00001 +
    pl.col("credits") * 0.25 +
    pl.col("debits") * 0.25 +
    (pl.col("txn_count") > 3).cast(pl.Float64) * 0.5
).alias("suspicion_score")
)

sessions.select(
    ["account_id","session_id","txn_count","total_amount","suspicion_score"]
).head()

account_id,session_id,txn_count,total_amount,suspicion_score
str,i32,u32,f64,f64
"""ACCT_170125""",211,3,2009.6,2.120096
"""ACCT_002877""",591,1,4298.48,0.7429848
"""ACCT_165595""",192,7,205434.48,7.4543448
"""ACCT_175795""",151,1,75100.0,1.451
"""ACCT_179899""",284,20,248167.74,16.981677


In [54]:
best_windows = (
    sessions
    .sort("suspicion_score", descending=True)
    .group_by("account_id")
    .head(1)
)

best_windows.head()

account_id,session_id,start,end,txn_count,total_amount,credits,debits,suspicion_score
str,i32,datetime[μs],datetime[μs],u32,f64,u32,u32,f64
"""ACCT_188808""",1,2025-05-12 03:42:58,2025-06-29 20:10:00,354,2.5131e7,147,207,499.606782
"""ACCT_085555""",1,2025-06-11 04:54:02,2025-06-26 11:59:00,1713,4.7305e7,801,912,1672.647294
"""ACCT_104881""",1,2024-11-11 08:49:19,2025-06-29 20:04:06,2437,1.6427e8,1133,1304,3349.078777
"""ACCT_092414""",145,2024-05-03 21:34:16,2024-05-04 15:44:49,2,6.1240e6,1,1,62.639562
"""ACCT_180113""",53,2023-02-06 18:00:09,2023-03-03 14:38:48,86,1.6402e6,32,54,77.102241


In [55]:
best_pd = best_windows.select([
    "account_id",
    "start",
    "end"
]).to_pandas()

submission = submission.merge(
    best_pd,
    on="account_id",
    how="left"
)

submission["suspicious_start"] = submission["start"]
submission["suspicious_end"] = submission["end"]

submission = submission.drop(columns=["start","end"])

submission.head()

,account_id,is_mule,suspicious_start,suspicious_end
0,ACCT_000005,0.002758,NaT,NaT
1,ACCT_000007,0.002758,NaT,NaT
2,ACCT_000009,0.002779,NaT,NaT
3,ACCT_000015,0.002758,NaT,NaT
4,ACCT_000016,0.002758,NaT,NaT


In [56]:
submission[submission["suspicious_start"].notna()].head()

,account_id,is_mule,suspicious_start,suspicious_end
43,ACCT_000116,0.666080,2021-06-06 16:15:02,2021-06-17 07:54:55
92,ACCT_000236,0.884386,2025-06-04 17:35:48,2025-06-07 10:01:13
161,ACCT_000440,0.978155,2025-05-11 16:41:00,2025-05-29 21:22:39
177,ACCT_000484,0.395564,2024-02-12 10:02:06,2024-03-01 15:56:31
231,ACCT_000646,0.884386,2024-05-13 21:31:14,2024-05-17 14:41:56


In [57]:
submission.to_csv("submission_with_temporal_windows.csv", index=False)

print("Final submission saved.")

Final submission saved.


In [58]:
submission["suspicious_start"].notna().sum()

np.int64(1644)

In [59]:
submission[
    (submission["is_mule"] > 0.7) &
    (submission["suspicious_start"].notna())
].head()

,account_id,is_mule,suspicious_start,suspicious_end
92,ACCT_000236,0.884386,2025-06-04 17:35:48,2025-06-07 10:01:13
161,ACCT_000440,0.978155,2025-05-11 16:41:00,2025-05-29 21:22:39
231,ACCT_000646,0.884386,2024-05-13 21:31:14,2024-05-17 14:41:56
309,ACCT_000866,0.782901,2023-10-31 03:40:17,2023-11-02 20:56:23
348,ACCT_000990,0.961116,2024-12-28 12:00:00,2025-01-01 01:16:46


In [60]:
submission[
    (submission["is_mule"] > 0.7) &
    (submission["suspicious_start"].isna())
].head()

,account_id,is_mule,suspicious_start,suspicious_end
1070,ACCT_003281,0.884386,NaT,NaT
1590,ACCT_004974,0.884386,NaT,NaT
7064,ACCT_022184,0.782901,NaT,NaT
10550,ACCT_033209,0.978155,NaT,NaT
11066,ACCT_034824,0.884386,NaT,NaT


In [61]:
txn_basic = pl.read_parquet("../outputs/features_txn_basic.parquet")

fallback_windows = (
    txn_basic
    .select([
        "account_id",
        pl.col("first_txn_date").alias("fallback_start"),
        pl.col("last_txn_date").alias("fallback_end")
    ])
    .to_pandas()
)

submission = submission.merge(
    fallback_windows,
    on="account_id",
    how="left"
)

mask = (
    (submission["is_mule"] > 0.7) &
    (submission["suspicious_start"].isna())
)

submission.loc[mask, "suspicious_start"] = submission.loc[mask, "fallback_start"]
submission.loc[mask, "suspicious_end"] = submission.loc[mask, "fallback_end"]

submission = submission.drop(columns=["fallback_start","fallback_end"])

In [62]:
submission["suspicious_start"].notna().sum()

np.int64(1644)

In [63]:
high_risk = submission[submission["is_mule"] > 0.7]["account_id"]

txn_basic = pl.read_parquet("../outputs/features_txn_basic.parquet")

accounts_with_txns = set(txn_basic["account_id"].to_list())
high_risk_set = set(high_risk)

missing_txn_accounts = high_risk_set - accounts_with_txns

print("High risk accounts:", len(high_risk_set))
print("High risk with transactions:", len(high_risk_set & accounts_with_txns))
print("High risk WITHOUT transactions:", len(missing_txn_accounts))

High risk accounts: 932
High risk with transactions: 932
High risk WITHOUT transactions: 0


In [64]:
submission[submission["suspicious_start"].notna()].head()

,account_id,is_mule,suspicious_start,suspicious_end
43,ACCT_000116,0.666080,2021-06-06 16:15:02,2021-06-17 07:54:55
92,ACCT_000236,0.884386,2025-06-04 17:35:48,2025-06-07 10:01:13
161,ACCT_000440,0.978155,2025-05-11 16:41:00,2025-05-29 21:22:39
177,ACCT_000484,0.395564,2024-02-12 10:02:06,2024-03-01 15:56:31
231,ACCT_000646,0.884386,2024-05-13 21:31:14,2024-05-17 14:41:56


In [65]:
best_windows.head()
print(best_windows.shape)

(1678, 9)


In [66]:
submission.columns

Index(['account_id', 'is_mule', 'suspicious_start', 'suspicious_end'], dtype='str')

In [67]:
print(submission["account_id"].dtype)
print(best_windows["account_id"].dtype)

str
String


In [68]:
best_pd = best_windows.select([
    "account_id",
    "start",
    "end"
]).to_pandas()

best_pd["account_id"] = best_pd["account_id"].astype(str)
submission["account_id"] = submission["account_id"].astype(str)

submission = submission.merge(
    best_pd,
    on="account_id",
    how="left"
)

submission["suspicious_start"] = submission["start"]
submission["suspicious_end"] = submission["end"]

submission = submission.drop(columns=["start","end"])

In [69]:
submission["suspicious_start"].notna().sum()

np.int64(1644)

In [70]:
submission.isnull().sum()

account_id              0
is_mule                 0
suspicious_start    62418
suspicious_end      62418
dtype: int64

In [71]:
submission["suspicious_start"] = submission["suspicious_start"].astype(str).str.replace(" ", "T")
submission["suspicious_end"] = submission["suspicious_end"].astype(str).str.replace(" ", "T")

In [72]:
submission.to_csv("submission_final.csv", index=False)

In [73]:
print("Rows:", submission.shape[0])
print("Columns:", submission.shape[1])

print("Max probability:", submission["is_mule"].max())

print("Accounts >0.5:", (submission["is_mule"]>0.5).sum())
print("Accounts >0.7:", (submission["is_mule"]>0.7).sum())

print("Temporal windows:", submission["suspicious_start"].notna().sum())

Rows: 64062
Columns: 4
Max probability: 0.999999
Accounts >0.5: 1271
Accounts >0.7: 932
Temporal windows: 1644


In [74]:
max_p = submission["is_mule"].max()
min_p = submission["is_mule"].min()

submission["is_mule"] = (submission["is_mule"] - min_p) / (max_p - min_p)

In [75]:
print("Rows:", submission.shape[0])
print("Columns:", submission.shape[1])

print("Max probability:", submission["is_mule"].max())

print("Accounts >0.3:", (submission["is_mule"]>0.3).sum())
print("Accounts >0.5:", (submission["is_mule"]>0.5).sum())
print("Accounts >0.7:", (submission["is_mule"]>0.7).sum())

print("Temporal windows:", submission["suspicious_start"].notna().sum())

Rows: 64062
Columns: 4
Max probability: 1.0
Accounts >0.3: 1611
Accounts >0.5: 1271
Accounts >0.7: 932
Temporal windows: 1644


In [76]:
submission["is_mule"] = submission["is_mule"].clip(1e-6, 1-1e-6)

In [77]:
submission.to_csv("submission_final.csv", index=False)

In [78]:
submission["suspicious_start"].notna().sum()

np.int64(1644)

In [79]:
print("FINAL CHECK")

print("Rows:", len(submission))
print("Unique accounts:", submission["account_id"].nunique())
print("Min prob:", submission["is_mule"].min())
print("Max prob:", submission["is_mule"].max())

print("\nThreshold distribution:")
for t in [0.1,0.2,0.3,0.5,0.7]:
    print(t, (submission["is_mule"] > t).sum())

print("\nTemporal windows:", submission["suspicious_start"].notna().sum())
print("Null starts:", submission["suspicious_start"].isna().sum())

FINAL CHECK
Rows: 64062
Unique accounts: 64062
Min prob: 1e-06
Max prob: 0.999999

Threshold distribution:
0.1 2654
0.2 1912
0.3 1611
0.5 1271
0.7 932

Temporal windows: 1644
Null starts: 62418


In [80]:
submission["is_mule"].describe()

count    64062.000000
mean         0.024707
std          0.118416
min          0.000001
25%          0.002757
50%          0.002757
75%          0.002923
max          0.999999
Name: is_mule, dtype: float64

In [81]:
top_1 = submission.sort_values("is_mule", ascending=False).head(1000)
top_5 = submission.sort_values("is_mule", ascending=False).head(5000)

print("Mean prob top 1000:", top_1["is_mule"].mean())
print("Mean prob top 5000:", top_5["is_mule"].mean())
print("Global mean:", submission["is_mule"].mean())

Mean prob top 1000: 0.8576347064197506
Mean prob top 5000: 0.28002559927772086
Global mean: 0.024706945939672088


In [82]:
import numpy as np

for pct in [0.01,0.02,0.05,0.1]:
    k = int(len(submission)*pct)
    mean_prob = submission.sort_values("is_mule", ascending=False).head(k)["is_mule"].mean()
    print(f"Top {int(pct*100)}% mean prob:", mean_prob)

Top 1% mean prob: 0.9120447096109118
Top 2% mean prob: 0.8047614438945142
Top 5% mean prob: 0.42664196910900887
Top 10% mean prob: 0.21976579589288509


In [83]:
for t in [0.2,0.3,0.4,0.5]:
    print(t, (submission["is_mule"]>t).sum())

0.2 1912
0.3 1611
0.4 1301
0.5 1271


In [84]:
print(sessions.columns)

['account_id', 'session_id', 'start', 'end', 'txn_count', 'total_amount', 'credits', 'debits', 'suspicion_score']
